# Lab 7 — Stemming vs Lemmatization
*NLP · answers notebook*

In [1]:
import pandas as pd
import nltk
# Run once if missing:
# nltk.download("wordnet"); nltk.download("omw-1.4")
# nltk.download("punkt_tab"); nltk.download("averaged_perceptron_tagger_eng")
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag, word_tokenize

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# From Lab 6, Exercise 2
POS_MAP = {
    "NN": "n", "NNS": "n", "NNP": "n", "NNPS": "n",
    "VB": "v", "VBD": "v", "VBG": "v", "VBN": "v", "VBP": "v", "VBZ": "v",
    "JJ": "a", "JJR": "a", "JJS": "a",
    "RB": "r", "RBR": "r", "RBS": "r",
}

def pos_aware_lemmatize(sentence):
    tagged = pos_tag(word_tokenize(sentence))
    return [(tok, tag, lemmatizer.lemmatize(tok.lower(), POS_MAP.get(tag, "n")))
            for tok, tag in tagged]

## Exercise 1 — Direct, word-by-word comparison

In [2]:
# Most plausible POS for each word, as the lab specifies
words = [("studies", "n"), ("studying", "v"), ("mice", "n"), ("better", "a"),
         ("running", "v"), ("ponies", "n"), ("was", "v")]

table1 = pd.DataFrame(
    [(w, stemmer.stem(w), lemmatizer.lemmatize(w, pos=p)) for w, p in words],
    columns=["Word", "Stem (Porter)", "Lemma (POS-aware)"],
).set_index("Word")
table1

,Stem (Porter),Lemma (POS-aware)
Word,,
studies,studi,study
studying,studi,study
mice,mice,mouse
better,better,good
running,run,run
ponies,poni,pony
was,wa,be


**1.2 Stems that are NOT valid English words:** `studi` (from *studies* and *studying*), `poni` (from *ponies*) and `wa` (from *was*). All the other stems (`mice`, `better`, `run`) are real words, but note that `mice` and `better` weren't reduced at all.

**1.3 Identical vs different:**
- **Identical:** only `running` → `run` (stem = lemma). `mice` and `better` also look "the same" in that the stem equals the word itself, but the lemma differs (`mouse`, `good`), so they count as different.
- **Noticeably different:** `studies` (`studi` / `study`), `studying` (`studi` / `study`), `mice` (`mice` / `mouse`), `better` (`better` / `good`), `ponies` (`poni` / `pony`), `was` (`wa` / `be`).

**1.4 Why a lemmatizer can get *mice* right and a stemmer can't:** *mice → mouse* is an **irregular** plural: the stem vowel changes (`ou` → `i`) and the ending is replaced, so there is no suffix to strip. A stemmer is a purely **string-rewriting** system: it only removes or replaces endings by rules such as `-ies → -i` or `-ing → ""`. Nothing in `mice` matches a suffix rule, and no rule can know that `mice` is related to `mouse`; that is a fact about the language, not about letter patterns. The WordNet lemmatizer has a **dictionary plus an exception list** of irregular forms (`mice → mouse`, `geese → goose`, `better → good`, `was → be`), so it can simply look the form up and return the headword.

### Questions
1. **Simpler to implement and run:** **stemming**. It is a handful of rewriting rules, needs no dictionary, no POS tagger and no context, and is fast.
2. **Which attempts a valid linguistic base form:** **lemmatization**. It returns the dictionary headword (the *lemma*), checked against WordNet.
3. **Which can produce non-words, and why by design:** **stemming** (`studi`, `poni`, `wa`). A stemmer only cares that related word forms collapse to the *same string*; it never checks a lexicon, so the output doesn't have to be a word. The goal is a consistent index key for matching, not a readable word.
4. **When they agreed:** only `running` → `run` in this table. They agree when the inflection is **regular and simple** (a plain suffix like `-ing` that is removed without changing the spelling of the base), because then suffix stripping and dictionary lookup arrive at the same string. For irregular forms (`mice`, `better`, `was`) or spelling changes (`-ies → y`) they diverge.

## Exercise 2 — Vocabulary reduction on a small corpus

**2.1 – 2.3 Tokenize, stem and lemmatize:**

In [3]:
corpus = [
    "Students study computer science.",
    "The students are studying computer science.",
    "A student studied computer science.",
]

tokens, stems, lemmas = [], [], []
for sent in corpus:
    for tok, tag, lemma in pos_aware_lemmatize(sent):
        tok = tok.lower()
        if not tok.isalpha():          # drop punctuation
            continue
        tokens.append(tok)
        stems.append(stemmer.stem(tok))
        lemmas.append(lemma)

pd.DataFrame({"token": tokens, "stem": stems, "lemma": lemmas}).drop_duplicates().reset_index(drop=True)

,token,stem,lemma
0,students,student,student
1,study,studi,study
2,computer,comput,computer
3,science,scienc,science
4,the,the,the
5,are,are,be
6,studying,studi,study
7,a,a,a
8,student,student,student
9,studied,studi,study


In [4]:
v0, v_stem, v_lem = len(set(tokens)), len(set(stems)), len(set(lemmas))

summary = pd.DataFrame({
    "vocabulary size": [v0, v_stem, v_lem],
    "reduction (%)": [0.0, round(100*(v0-v_stem)/v0, 1), round(100*(v0-v_lem)/v0, 1)],
}, index=["Original", "After Porter stemming", "After POS-aware lemmatization"])
print("Original vocab :", sorted(set(tokens)))
print("Stemmed vocab  :", sorted(set(stems)))
print("Lemma vocab    :", sorted(set(lemmas)))
summary

Original vocab : ['a', 'are', 'computer', 'science', 'student', 'students', 'studied', 'study', 'studying', 'the']
Stemmed vocab  : ['a', 'are', 'comput', 'scienc', 'student', 'studi', 'the']
Lemma vocab    : ['a', 'be', 'computer', 'science', 'student', 'study', 'the']


,vocabulary size,reduction (%)
Original,10,0.0
After Porter stemming,7,30.0
After POS-aware lemmatization,7,30.0


**2.4 Results**

| | vocabulary size | reduction |
|---|---|---|
| Original | 10 | – |
| After Porter stemming | 7 | 30 % |
| After POS-aware lemmatization | 7 | 30 % |

Both methods shrink the vocabulary from 10 to 7 types, so the reduction is the same (30 %) **but the content is different**: the stemmed vocabulary contains the non-words `comput`, `scienc` and `studi`, whereas the lemma vocabulary contains only real words (`computer`, `science`, `study`, `be`, `student`). Lemmatization also maps `are` to `be`, which the stemmer leaves untouched (`are`); the stemmer in turn merges nothing extra, since `computer`/`science` just lose their endings.

### Question — benefit and risk of vocabulary reduction
**Benefit:** in a bag-of-words classifier trained on little data, each distinct word form gets its own feature and its own weight. If *student*, *students*, *studying* and *studied* are separate features, each appears only a few times, so the model can't learn reliable weights (data **sparsity**). Merging them pools the counts, giving fewer, better-estimated features, less over-fitting, a smaller model, and better generalisation to unseen inflected forms.

**Risk:** too much reduction **loses information**. Over-aggressive conflation merges words with different meanings (*university/universe* in Lab 5, *organ/organization*), and removing inflection can erase useful signals (tense, number, negation like *was* vs *is*, or *shall* vs *should*). For tasks where these distinctions matter (sentiment, legal text, generation) reduction can hurt accuracy.

## Exercise 3 — Choosing the right technique

**3.1 / 3.4 Decisions**

| Application | Recommended technique | One-sentence justification |
|---|---|---|
| Search engine | **Stemming** (lemmatization is a fine alternative) | It conflates *study/studies/studying* to one key for higher recall; the stem never has to be shown to the user, and it's fast and needs no tagger. |
| Chatbot generating responses | **Lemmatization** (for analysing user input) and **neither** for the generated text itself | The output must be valid, grammatical words, and a lemma is a real word (`study`) that can then be re-inflected, whereas `studi` is not a word. |
| Large-scale document clustering | **Stemming** | Millions of documents need the cheapest option; stemming needs no POS tagging and gives the same 30 % reduction as lemmatization in our test. |
| Legal-document analysis | **Neither** | Exact word forms like *shall* vs *should* carry legal meaning, and both techniques merge or alter them. |

**3.2 Recall in the search engine.** Suppose a user searches for "study" and the corpus has documents containing only *studies* or only *studying*. Without normalisation, the query term `study` matches only documents with that exact form, so the others are missed even though they are relevant (**false negatives**). Recall = relevant documents found / all relevant documents, so these misses lower it. If *study*, *studies* and *studying* are all reduced to `studi` at indexing time and at query time, all three kinds of document match the query, so recall goes up (with a possible small cost in precision from over-stemming).

**3.3 Why `studi` is unacceptable in a chatbot reply.** A chatbot's output is read by a person: "I am studi for my exam" is ungrammatical and not even a word, which makes the bot look broken and can confuse the user. Stems are internal matching keys, not surface forms. Generation requires valid words with the correct inflection (`studying`), which a lemma plus a morphological generator can provide but a truncated stem cannot.